# Project 2

Author: Evan Whitfield

Course: ST554 - Spring 2026

Purpose: Project 2

Necessary imports for Part 1 of the project below.

In [2]:
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
import Project2 as pro2

The below code was necessary to run instead of having to refresh the kernel after I made changes to the script file.

In [3]:
import importlib

importlib.reload(pro2)

<module 'Project2' from '/home/jupyter-edwhitfi@ncsu.edu/Project2.py'>

## Data Read-in (From CSV)

Setting up the spark session and reading in the data from the .csv file air.csv.

In [4]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

air_data = pro2.SparkDataChecker.read_csv(spark, 'air.csv')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 23:33:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 23:33:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/26 23:33:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Check-In Range Function Testing

Below we will be testing the various methods in our script, starting with the `check-in-range` method.

In [5]:
air_data.check_in_range("T")

Must give at least one of lower or upper.


`check_in_range` fails if there is not a range to check.

In [6]:
air_data.check_in_range("Date", lower = 11)

Column 'Date' is not numeric.


`check_in_range` fails if the column is not numeric. User must give a column with a numeric type.

In [7]:
air_data.check_in_range("CO(GT)", lower = 2.1).show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|CO(GT)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|          false|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|  

26/03/26 23:33:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-edwhitfi@ncsu.edu/air.csv


`check_in_range` shows us when CO(GT) is 2.1 or above. These could potentially be considered "high" CO levels.

In [8]:
air_data.check_in_range("CO(GT)", lower = 1.5, upper = 2.1).show()

26/03/26 23:33:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-edwhitfi@ncsu.edu/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|CO(GT)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|          false|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|  

`check_in_range` shows us when CO(GT) is between 1.5 and 2.1. These could be potentially be considered "medium" CO levels.

In [9]:
air_data.check_in_range("CO(GT)",  upper = 1.49).show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|CO(GT)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|          false|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|          false|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|  

26/03/26 23:33:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-edwhitfi@ncsu.edu/air.csv


`check_in_range` shows us when CO(GT) is below 1.5. These could be considered potentially "low" CO levels.

In [10]:
air_data.summarize_min_max("CO(GT)", groupedby = 'Date')

,Date,min,max
0,9/2/2004,-200.0,5.2
1,12/26/2004,0.3,3.8
2,2/18/2005,-200.0,4.8
3,10/10/2004,-200.0,3.7
4,10/11/2004,0.4,4.8
...,...,...,...
386,1/23/2005,-200.0,0.8
387,6/28/2004,0.4,5.6
388,8/16/2004,0.3,2.1
389,12/20/2004,0.6,4.1


Here, we grouped the data by 'Date' and looked at the minimum and maximum CO levels for the dates in our dataset. In the future, -200 values should be removed from the consideration as they are considered missing values. As it currently is, there is no way to see the minimum CO reading for 9/2/2004 because there was a missing CO reading from that date that is being shown as the minimum value.

In [11]:
air_data.summarize_min_max(groupedby = 'Date')

26/03/26 23:34:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/26 23:34:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date
 Schema: _c0, Date
Expected: _c0 but found: 
CSV file: file:///home/jupyter-edwhitfi@ncsu.edu/air.csv


,Date,_c0_min,_c0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,...,PT08.S4(NO2)_min,PT08.S4(NO2)_max,PT08.S5(O3)_min,PT08.S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,9/2/2004,4206,4229,-200.0,5.2,832,1464,-200,-200,3.1,...,1312,2366,738,1738,21.0,38.7,17.3,57.2,1.1729,1.5165
1,12/26/2004,6966,6989,0.3,3.8,848,1423,-200,-200,0.7,...,1062,1531,438,1381,10.4,14.9,57.9,82.9,0.9640,1.1381
2,2/18/2005,8262,8285,-200.0,4.8,872,1288,-200,-200,0.7,...,794,1333,350,1502,5.4,11.8,30.9,52.9,0.4223,0.4922
3,10/10/2004,5118,5141,-200.0,3.7,941,1520,-200,-200,3.6,...,1406,1812,549,1357,19.4,24.8,56.9,80.5,1.6507,1.8881
4,10/11/2004,5142,5165,0.4,4.8,825,1464,-200,-200,1.4,...,1302,2072,494,1625,17.9,23.2,52.0,77.7,1.3032,1.6384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386,1/23/2005,7638,7661,-200.0,0.8,849,1174,-200,-200,1.6,...,810,1220,737,1445,2.5,8.9,36.0,78.2,0.3861,0.7074
387,6/28/2004,2622,2645,0.4,5.6,827,1441,-200,-200,2.7,...,1438,2568,619,1791,22.8,40.4,19.0,50.1,1.3447,1.4854
388,8/16/2004,3798,3821,0.3,2.1,749,1028,-200,-200,1.5,...,1244,1624,458,1135,21.3,39.7,14.6,47.0,0.9295,1.3722
389,12/20/2004,6822,6845,0.6,4.1,762,1185,-200,-200,0.7,...,783,1250,454,1104,6.8,9.8,32.4,82.8,0.3340,0.8561


Again, we grouped by Date, but without a given numeric column, the program returns a min and a max for every numeric column. Again, any missing value is set to -200, which appears in the min category. Data cleaning is necessary to return the proper minimum value recorded for the given Date.

In [12]:
air_data.summarize_counts("Date")

,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


Here, we wanted to see if the date column was recognized as a string column, and if so, how many recordings were made for each date. It appears that a recording was made every hour of each day.

In [13]:
air_data.summarize_counts("Date", "T")

'T' is not a string column.


,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


In [14]:
air_data.summarize_counts("T", "Date")

'T' is not a string column.


,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


In [15]:
air_data.summarize_counts("T")

'T' is not a string column.


We wanted to run summarize_counts with column "T" a few times to verify the correct results. Since "T" is for temperature (and should be numeric), the method tells us that the column is not a string column, and therefore cannot be used in this particular method.

## Reading in Data from pandas.

Below we read in the data via and a pandas dataframe. We then verified that the `check_in_range` method still worked properly.

In [17]:
df = pd.read_csv('air.csv')
air_data2 = pro2.SparkDataChecker.from_pandas(spark, df)

air_data2.check_in_range("T", lower = 10, upper = 12).show()

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_in_range|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|     false|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|     false|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|      true|
|         

The data was read into the class correctly. The method shows us when the temperature is between 10 and 12. I believe that these are Celsius temperature values. I was looking for when the temperature might be near freezing, but the column would have returned false for all responses. This would not have verified the method worked, so I went for the range of 10 to 12 instead. 

## PART 2

Now we will read in some weekly NFL data to do some data exploration using Spark.

In [18]:
import pyspark.pandas as ps

spark.conf.set("spark.sql.ansi.enabled", "false")

nfl_data = ps.read_csv("weekly_nfl_data.csv")

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [19]:
nfl_data.head(5)

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Here we can see that the data was read into our pandas on spark dataframe correctly despitre the read message. I honestly do not know what it is referencing at this time, but the dataframe seems to be intact. We will proceed with caution.

In [20]:
for col in nfl_data.columns:
    print(col)

player_id
player_name
player_display_name
position
position_group
headshot_url
recent_team
season
week
season_type
opponent_team
completions
attempts
passing_yards
passing_tds
interceptions
sacks
sack_yards
sack_fumbles
sack_fumbles_lost
passing_air_yards
passing_yards_after_catch
passing_first_downs
passing_epa
passing_2pt_conversions
pacr
dakota
carries
rushing_yards
rushing_tds
rushing_fumbles
rushing_fumbles_lost
rushing_first_downs
rushing_epa
rushing_2pt_conversions
receptions
targets
receiving_yards
receiving_tds
receiving_fumbles
receiving_fumbles_lost
receiving_air_yards
receiving_yards_after_catch
receiving_first_downs
receiving_epa
receiving_2pt_conversions
racr
target_share
air_yards_share
wopr
special_teams_tds
fantasy_points
fantasy_points_ppr


Here are all of the columns that are datafram contains. That is a whole bunch! I do not know what several of them are, such as "racr" or "wopr" but thankfully we do not need them for our investigation.

We want to look at the QB position from 2005 to 2023, but only in the regular season. Since many of the stats do not typically apply to the QB position, they will not be considered going forward. Only some of the main passing statistics will be considered.

In [23]:
nfl_qb_reg = nfl_data.loc[(nfl_data["position"] == "QB") &
                      (nfl_data["season"] >= 2005) &
                      (nfl_data["season"] <= 2023) &
                      (nfl_data["season_type"] == "REG")
                      ,["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds","interceptions"]
                       ]

In [24]:
nfl_qb_reg.head(5)

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


Seems to have worked at this point. We only see a view names, but all of them have passing attempts, so they should be QBs.

In [25]:
nfl_qb_season = nfl_qb_reg.groupby(
    ["player_display_name", "season"]
).agg({
    "completions": ["sum", "mean"],
    "attempts": ["sum", "mean"],
    "passing_yards": ["sum", "mean"],
    "passing_tds": ["sum", "mean"],
    "interceptions": ["sum", "mean"]
})

In [26]:
nfl_qb_season.head(5)

completions            attempts            passing_yards             passing_tds           interceptions          
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                   
Jake Delhomme       2006           263  20.230769      431  33.153846        2805.0  215.769231          17  1.307692          11.0  0.846154
Jake Plummer        2005           277  17.312500      456  28.500000        3366.0  210.375000          18  1.125000           7.0  0.437500
Matt Schaub         2006            18   3.600000       27   5.400000         208.0   41.600000           1  0.200000           2.0  0.400000
Vince Young         2006           184  12.266667      356  23.733333        2199.0  146.600000          12  0.800000          13.0  0.866667
Kerry Collins       2007            50   8.333333       82  13.666667         531.0   88.500000           0  0.000000           0.0  0.000000

Now we can see the sum and average values of all of the main passing statistics from the QBs in the desired seasons. However, this will include every QB, even if they only attempted one pass the entire season. We should probably only look at QBs that meet a minimum requirement.

In [27]:
#Completion percentage = completions/attempts * 100 to turn it into a percentage
nfl_qb_season["completion_percentage"] = (
    nfl_qb_season["completions"]["sum"] / nfl_qb_season["attempts"]["sum"] * 100
)

#td/int ratio, but replacing 0 for ints with None so we aren't dividing by 0
nfl_qb_season["td_int_ratio"] = (
    nfl_qb_season["passing_tds"]["sum"] / nfl_qb_season["interceptions"]["sum"].replace(0, None)
)

In [29]:
nfl_qb_season.head(5)

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Jake Delhomme       2006           263  20.230769      431  33.153846        2805.0  215.769231          17  1.307692          11.0  0.846154             61.020882     1.545455
Jake Plummer        2005           277  17.312500      456  28.500000        3366.0  210.375000          18  1.125000           7.0  0.437500             60.745614     2.571429
Matt Schaub         2006            18   3.600000       27   5.400000         208.0   41.600000           1  0.200000           2.0  0.400000             66.666667     0.500000
Vince Young         2006           184  12.266667      356  23.733333        2199.0  146.600000          12  0.800000          13.0  0.866667             51.685393     0.923077
Kerry Collins       2007            50   8.333333       82  13.666667         531.0   88.500000           0  0.000000           0.0  0.000000             60.975610          NaN

Well, before we do that, lets create some new statistics for completion percentage and td/int ratio. Important stats for detecting good QB play!

In [30]:
nfl_qualified_qb_seasons = nfl_qb_season[nfl_qb_season[("attempts", "sum")] >= 50]

In [31]:
nfl_qualified_qb_seasons.sort_values(by="completion_percentage",ascending=False).head(40)

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
C.J. Beathard       2023            40   6.666667       53   8.833333         349.0   58.166667           1  0.166667           0.0  0.000000             75.471698          NaN
Colt McCoy          2021            74  10.571429       99  14.142857         740.0  105.714286           3  0.428571           1.0  0.142857             74.747475     3.000000
Matt Schaub         2019            50  10.000000       67  13.400000         580.0  116.000000           3  0.600000           1.0  0.200000             74.626866     3.000000
Drew Brees          2018           364  24.266667      489  32.600000        3992.0  266.133333          32  2.133333           5.0  0.333333             74.437628     6.400000
                    2019           281  25.545455      378  34.363636        2979.0  270.818182          27  2.454545           4.0  0.363636             74.338624     6.750000
Mason Rudolph       2023            55  13.750000       74  18.500000         719.0  179.750000           3  0.750000           0.0  0.000000             74.324324          NaN
Taysom Hill         2020            88   5.500000      121   7.562500         928.0   58.000000           4  0.250000           2.0  0.125000             72.727273     2.000000
Nick Foles          2018           141  28.200000      195  39.000000        1413.0  282.600000           7  1.400000           4.0  0.800000             72.307692     1.750000
Drew Brees          2017           386  24.125000      536  33.500000        4334.0  270.875000          23  1.437500           8.0  0.500000             72.014925     2.875000
Sam Bradford        2016           395  26.333333      552  36.800000        3877.0  258.466667          20  1.333333           5.0  0.333333             71.557971     4.000000
Drew Brees          2011           471  29.437500      660  41.250000        5535.0  345.937500          46  2.875000          14.0  0.875000             71.363636     3.285714
Colt McCoy          2014            91  18.200000      128  25.600000        1057.0  211.400000           4  0.800000           3.0  0.600000             71.093750     1.333333
Aaron Rodgers       2020           372  23.250000      526  32.875000        4299.0  268.687500          48  3.000000           5.0  0.312500             70.722433     9.600000
Bailey Zappe        2022            65  16.250000       92  23.000000         781.0  195.250000           5  1.250000           3.0  0.750000             70.652174     1.666667
Drew Brees          2009           363  24.200000      514  34.266667        4388.0  292.533333          34  2.266667          11.0  0.733333             70.622568     3.090909
                    2020           275  22.916667      390  32.500000        2942.0  245.166667          24  2.000000           6.0  0.500000             70.512821     4.000000
Joe Burrow          2021           366  22.875000      520  32.500000        4611.0  288.187500          34  2.125000          14.0  0.875000             70.384615     2.428571
Derek Carr          2019           361  22.562500      513  32.062500        4054.0  253.375000          21  1.312500           8.0  0.500000             70.370370     2.625000
Jake Browning       2023           171  19.000000      243  27.000000        1936.0  215.111111          12  1.333333           7.0  0.777778             70.370370     1.714286
Chase Daniel        2019            45  15.000000       64  21.333333         435.0  145.000000           3  1.000000           2.0  

We decided that the minimum statistic should be that the attempted at least 50 passes. Here, we sorted the QBs by completion percentage and looked at the top 40 seasons. Interestingly enough, Patrick Mahommes is not in this list, but Drew Brees is on here several times!

In [32]:
nfl_qualified_qb_seasons.sort_values(by="td_int_ratio", ascending=False).head(40)

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Tom Brady           2016           291  24.250000      432  36.000000        3554.0  296.166667          28  2.333333           2.0  0.166667             67.361111    14.000000
Nick Foles          2013           203  15.615385      317  24.384615        2891.0  222.384615          27  2.076923           2.0  0.153846             64.037855    13.500000
Josh McCown         2013           149  18.625000      224  28.000000        1829.0  228.625000          13  1.625000           1.0  0.125000             66.517857    13.000000
Aaron Rodgers       2018           372  23.250000      597  37.312500        4442.0  277.625000          25  1.562500           2.0  0.125000             62.311558    12.500000
Damon Huard         2006           148  14.800000      244  24.400000        1878.0  187.800000          11  1.100000           1.0  0.100000             60.655738    11.000000
Aaron Rodgers       2020           372  23.250000      526  32.875000        4299.0  268.687500          48  3.000000           5.0  0.312500             70.722433     9.600000
                    2021           366  22.875000      531  33.187500        4115.0  257.187500          37  2.312500           4.0  0.250000             68.926554     9.250000
Tom Brady           2010           324  20.250000      492  30.750000        3900.0  243.750000          36  2.250000           4.0  0.250000             65.853659     9.000000
Jake Delhomme       2007            55  18.333333       86  28.666667         617.0  205.666667           8  2.666667           1.0  0.333333             63.953488     8.000000
Aaron Rodgers       2014           341  21.312500      520  32.500000        4381.0  273.812500          38  2.375000           5.0  0.312500             65.576923     7.600000
                    2011           342  22.800000      501  33.400000        4636.0  309.066667          45  3.000000           6.0  0.400000             68.263473     7.500000
Drew Brees          2019           281  25.545455      378  34.363636        2979.0  270.818182          27  2.454545           4.0  0.363636             74.338624     6.750000
Aaron Rodgers       2019           353  22.062500      569  35.562500        4002.0  250.125000          26  1.625000           4.0  0.250000             62.038664     6.500000
Drew Brees          2018           364  24.266667      489  32.600000        3992.0  266.133333          32  2.133333           5.0  0.333333             74.437628     6.400000
Patrick Mahomes     2020           390  26.000000      588  39.200000        4740.0  316.000000          38  2.533333           6.0  0.400000             66.326531     6.333333
Tom Brady           2007           398  24.875000      578  36.125000        4806.0  300.375000          50  3.125000           8.0  0.500000             68.858131     6.250000
Russell Wilson      2019           341  21.312500      516  32.250000        4110.0  256.875000          31  1.937500           5.0  0.312500             66.085271     6.200000
David Garrard       2007           208  17.333333      325  27.083333        2509.0  209.083333          18  1.500000           3.0  0.250000             64.000000     6.000000
Matthew Stafford    2010            57  14.250000       96  24.000000         535.0  133.750000           6  1.500000           1.0  0.250000             59.375000     6.000000
Lamar Jackson       2019           265  17.666667      401  26.733333        3127.0  208.466667          36  2.400000           6.0  

Here, we decided to sort by TD/INT ratio. Patrick Mahomes made this list, but only after our boy Jake Delhomme. Ignore the fact that he only have 8 TDs and 1 INT that season, he is a Panthers legend.

### Spark SQL Dataframe

Now it is time to do it again, but this time using Spark SQL. This seemed much easier to put together in my humble opinion. Maybe it is recentcy bias creeping in some.

In [34]:
spark2 = SparkSession.builder.master("loc[2]").appName("my_app2").getOrCreate()

nfl_data_2 = spark.read.csv("weekly_nfl_data.csv", header=True, inferSchema=True)

In [40]:
nfl_qb_season_2 = (
    nfl_data_2.filter(
        (F.col("position") == "QB") &
        (F.col("season") >= 2005) &
        (F.col("season") <= 2023) &
        (F.col("season_type") == "REG")
    )
    .select(
        "player_display_name",
        "season",
        "week",
        "completions",
        "attempts",
        "passing_yards",
        "passing_tds",
        "interceptions"
    )
    .groupBy("player_display_name", "season")
    .agg(
        F.sum("completions").alias("season_comps"),
        F.round(F.mean("completions"),2).alias("comps_pg"),
        F.sum("attempts").alias("season_attempts"),
        F.round(F.mean("attempts"),2).alias("attempts_pg"),
        F.sum("passing_yards").alias("season_pass_yd"),
        F.round(F.mean("passing_yards"),2).alias("pass_yd_pg"),
        F.sum("passing_tds").alias("season_pass_tds"),
        F.round(F.mean("passing_tds"),2).alias("pass_tds_pg"),
        F.sum("interceptions").alias("season_ints"),
        F.round(F.mean("interceptions"),2).alias("ints_pg")
    )
    .withColumn(
        "comp_perc",
        F.round(F.col("season_comps") / F.col("season_attempts") * 100,2)
    )
    .withColumn(
        "td_int_ratio",
        F.when(F.col("season_ints") == 0, None)
         .otherwise(F.round(F.col("season_pass_tds") / F.col("season_ints"),2))
    )
)

In [41]:
nfl_qb_season_2.filter(F.col("season_attempts") >= 50).orderBy(F.col("td_int_ratio").desc()).show(40)

+-------------------+------+------------+--------+---------------+-----------+--------------+----------+---------------+-----------+-----------+-------+---------+------------+
|player_display_name|season|season_comps|comps_pg|season_attempts|attempts_pg|season_pass_yd|pass_yd_pg|season_pass_tds|pass_tds_pg|season_ints|ints_pg|comp_perc|td_int_ratio|
+-------------------+------+------------+--------+---------------+-----------+--------------+----------+---------------+-----------+-----------+-------+---------+------------+
|          Tom Brady|  2016|         291|   24.25|            432|       36.0|        3554.0|    296.17|             28|       2.33|        2.0|   0.17|    67.36|        14.0|
|         Nick Foles|  2013|         203|   15.62|            317|      24.38|        2891.0|    222.38|             27|       2.08|        2.0|   0.15|    64.04|        13.5|
|        Josh McCown|  2013|         149|   18.63|            224|       28.0|        1829.0|    228.63|             13|

Yuck! That is hard to read. I tried to round the values and be creative with my alias to try to make it fit nicely on the screen, but I would need to shorten the column names even further. Good thing is that it worked, and it seemed to be in the same order as a pandas on Spark list.

In [42]:
nfl_qb_season_2.filter(F.col("season_attempts") >= 50).orderBy(F.col("td_int_ratio").desc()).show(40)

+-------------------+------+------------+--------+---------------+-----------+--------------+----------+---------------+-----------+-----------+-------+---------+------------+
|player_display_name|season|season_comps|comps_pg|season_attempts|attempts_pg|season_pass_yd|pass_yd_pg|season_pass_tds|pass_tds_pg|season_ints|ints_pg|comp_perc|td_int_ratio|
+-------------------+------+------------+--------+---------------+-----------+--------------+----------+---------------+-----------+-----------+-------+---------+------------+
|          Tom Brady|  2016|         291|   24.25|            432|       36.0|        3554.0|    296.17|             28|       2.33|        2.0|   0.17|    67.36|        14.0|
|         Nick Foles|  2013|         203|   15.62|            317|      24.38|        2891.0|    222.38|             27|       2.08|        2.0|   0.15|    64.04|        13.5|
|        Josh McCown|  2013|         149|   18.63|            224|       28.0|        1829.0|    228.63|             13|

Here we can see the top 40 qualifying QBs by TD/INT ratio again. The order appears to be the same. However, I did something a little ahead knowing what might happen. The TD/INT ratio could potential cause someone to divide by 0 if the QB had zero ints. Different formats will handle this result differently. It if become a NaN, then it might accidently be repeated as maximum! Uh oh! Therefore, I asked the dataframe to have any 0 values just count as None. This seemed to make my lists work the same for both types of dataframes we looked at.